In [1]:
# Install necessary packages
!pip install transformers sentence_transformers ipywidgets flask datasets

# Enable ipywidgets
import sys
!jupyter nbextension enable --py widgetsnbextension

# Import libraries
import json
import logging
import torch
from transformers import BertTokenizer, BertForQuestionAnswering
from sentence_transformers import SentenceTransformer, util
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set up logging
logging.basicConfig(level=logging.INFO)

# Enable GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download the SQuAD v2.0 dataset
!wget -q https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json

# Load the SQuAD v2.0 dataset
with open('train-v2.0.json', 'r') as f:
    data = json.load(f)

# Extract contexts from the dataset
contexts = []
for article in data['data']:
    for paragraph in article['paragraphs']:
        contexts.append(paragraph['context'])

# Load pre-trained BERT model and tokenizer for QA
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')
model.to(device)  # Move model to GPU

# Load SBERT model for improved context matching
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Encode all contexts using SBERT for fast similarity search
context_embeddings = sbert_model.encode(contexts, convert_to_tensor=True, device=device, show_progress_bar=True)

def find_best_context(question):
    """Find the most relevant context using SBERT-based cosine similarity."""
    question_embedding = sbert_model.encode(question, convert_to_tensor=True, device=device)
    similarities = util.pytorch_cos_sim(question_embedding, context_embeddings).flatten()
    best_index = similarities.argmax().item()
    return contexts[best_index], similarities[best_index].item()  # Return context and its similarity score

def generate_answer(question, context):
    """Generate an answer using the BERT QA model."""
    # Tokenize with truncation to handle long sequences
    inputs = tokenizer(
        question,
        context,
        return_tensors='pt',
        max_length=512, # I added this due to BERT's limitation
        truncation=True  # Enable truncation
    ).to(device)
    input_ids = inputs['input_ids'].tolist()[0]

    with torch.no_grad():
        outputs = model(**inputs)

    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Find the most likely start and end of the answer span
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    # Confidence threshold
    confidence_threshold = 0.4

    # Check if the answer is valid
    start_score = answer_start_scores[0, answer_start].item()
    if start_score < confidence_threshold:
        return "I'm not confident about the answer. Can you please ask differently?"

    # Ensure that the end index is greater than the start index
    if answer_end <= answer_start:
        return "I'm sorry, but I couldn't find an answer in the context provided."

    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
    )

    # Check for coherence of the answer
    if not answer.strip():  # If the answer is empty
        return "I'm sorry, but I couldn't find an answer in the context provided."

    return answer

# Create a simple chat interface using ipywidgets
def chat_with_bot():
    output = widgets.Output()
    question_input = widgets.Text(
        description='You:',
        placeholder='Type your question here...'
    )

    display(question_input, output)

    def on_submit(change):
        user_input = change['new']
        if not user_input.strip():
            return

        # Find the most relevant context and its similarity score
        context, similarity_score = find_best_context(user_input)
        logging.info(f"User question: {user_input} | Best context similarity: {similarity_score:.2f}")

        # Generate the answer
        answer = generate_answer(user_input, context)

        # Clear the output and display the interaction
        with output:
            clear_output(wait=True)
            print(f"You: {user_input}")
            print(f"Bot: {answer}")

        # Clear the input for the next question
        question_input.value = ''

    question_input.observe(on_submit, names='value')

# Start the chat interface
chat_with_bot()

Enabling notebook extension jupyter-js-widgets/extension...
Paths used for configuration of notebook: 
    	/root/.jupyter/nbconfig/notebook.json
Paths used for configuration of notebook: 
    	
      - Validating: OK
Paths used for configuration of notebook: 
    	/root/.jupyter/nbconfig/notebook.json
Using device: cuda


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squa

Batches:   0%|          | 0/595 [00:00<?, ?it/s]

Text(value='', description='You:', placeholder='Type your question here...')

Output()

In [2]:
!pip install gradio
import gradio as gr

def respond(history, user_input):
    if not user_input.strip():
        history.append(("System", "Please enter a valid question."))
        return history, ""

    context, _ = find_best_context(user_input)
    answer = generate_answer(user_input, context)

    history.append((user_input, answer))
    return history, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Chat with BERT Bot")

    chatbot = gr.Chatbot()

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Type your question here...",
            label="Your Question"
        )
        send_button = gr.Button("Send")

    send_button.click(respond, inputs=[chatbot, user_input], outputs=[chatbot, user_input])
    user_input.submit(respond, inputs=[chatbot, user_input], outputs=[chatbot, user_input])

demo.launch(share=True)

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:223: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80d9500beec42ba879.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
